<a href="https://colab.research.google.com/github/shaktidharreddy/Agentic_AI_eduhubspot_aug_sep_2026/blob/main/ModuleGAI_3/Working_with_LLM_APIs_%26_Application_Frameworks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Lab 1: Build a multi-provider LLM gateway in Python that routes requests to OpenAI, Anthropic, or local Ollama based on task type**

### **1. Install Required Libraries**
We begin by installing and upgrading the official `openai` Python SDK. This package provides the interface needed to communicate with OpenAI's APIs and is critical for both the Multi-Provider Gateway (Lab 1) and Instructor (Lab 2).

In [1]:
# Use pip to quietly install or upgrade the openai python SDK
# This library handles connectivity and requests to the OpenAI developer platform
!pip -q install --upgrade openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 31.7 MB/s eta 0:00:00


### **2. Securely Retrieve the OpenAI API Key**
Using `google.colab.userdata`, we safely pull the API key stored in Colab's Secrets manager under the key name `'APIKEY'`. Storing keys in Secrets prevents accidental hardcoding of sensitive credentials.

In [3]:
import os
from google.colab import userdata

# Retrieve the secret 'APIKEY' stored securely in the Colab secrets manager panel
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

# Verify that the user has added their API key and enabled notebook access
if not OPENAI_API_KEY:
    raise ValueError("OpenAI API key not found in Colab Secrets. Add a secret named 'APIKEY' and enable notebook access.")

# Export the key to environment variables where typical client libraries expect it
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("OpenAI API key loaded successfully from Colab Secrets.")

OpenAI API key loaded successfully from Colab Secrets.


### **3. Import Required Modules**
Here, we import the core packages required for our gateway, including standard utility modules (`time`, `math`, `random`), structured data modeling helpers (`dataclasses`, `typing`), and the newly installed `OpenAI` client class.

In [4]:
import os
import time
import math
import random
from dataclasses import dataclass
from typing import Optional, Literal, Dict

# Import the main OpenAI client class to interface with the modern v1+ SDK
from openai import OpenAI

print("Libraries imported successfully.")

Libraries imported successfully.


### **4. Define Request and Result Models**
We define structured data containers using Python's `@dataclass`.
- `GatewayRequest`: Models the incoming prompt, the intended task type (e.g., chat, coding), stream preferences, and parameters like temperature.
- `GatewayResult`: Models the structured output returned by the gateway, storing execution metadata, routing info, calculated costs, and actual text output.

In [ ]:
# Establish a literal type union restricting tasks to 4 distinct categories
TaskType = Literal["chat", "coding", "analysis", "private_local"]

@dataclass
class GatewayRequest:
    """Dataclass holding all request configuration parameters passed to the gateway."""
    prompt: str
    task_type: TaskType = "chat"
    stream: bool = True
    max_retries: int = 3
    temperature: float = 0.2

@dataclass
class GatewayResult:
    """Dataclass summarizing the execution results, costs, and metadata for auditability."""
    provider_requested: str
    provider_used: str
    model: str
    text: str
    estimated_input_tokens: int
    estimated_output_tokens: int
    estimated_cost_usd: float
    attempts: int
    success: bool
    error: Optional[str] = None

### **5. Define the Routing Table**
We create a dictionary that routes specific task types (`chat`, `coding`, `analysis`, `private_local`) to their respective ideal providers (such as `openai`, `anthropic`, or `ollama`) and specifies which model should handle the query.

In [ ]:
# Define routing rule dictionary determining target provider/model for each task type
ROUTING_TABLE = {
    "chat": {"preferred_provider": "openai", "model": "gpt-4o-mini"},
    "coding": {"preferred_provider": "openai", "model": "gpt-4o-mini"},
    "analysis": {"preferred_provider": "anthropic", "model": "gpt-4o-mini"},
    "private_local": {"preferred_provider": "ollama", "model": "gpt-4o-mini"},
}

print("Routing table loaded.")
ROUTING_TABLE

Routing table loaded.


{'chat': {'preferred_provider': 'openai', 'model': 'gpt-4o-mini'},
 'coding': {'preferred_provider': 'openai', 'model': 'gpt-4o-mini'},
 'analysis': {'preferred_provider': 'anthropic', 'model': 'gpt-4o-mini'},
 'private_local': {'preferred_provider': 'ollama', 'model': 'gpt-4o-mini'}}

### **6. Token and Cost Estimation Helpers**
To estimate API usage cost dynamically:
- `estimate_tokens`: Employs a common heuristic rule (1 token ≈ 4 characters) to estimate token count when accurate tiktoken encoding is not required.
- `estimate_cost`: Looks up the current pricing matrix per million tokens for a given model and calculates estimated expenses for both input and output sequences.

In [ ]:
def estimate_tokens(text: str) -> int:
    """Estimate token count using the standard character-to-token heuristic (4 chars per token)."""
    if not text:
        return 0
    return max(1, math.ceil(len(text) / 4))

# Dynamic pricing mapping specifying input/output costs per 1 Million tokens
PRICING_PER_1M = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60}
}

def estimate_cost(model: str, input_text: str, output_text: str):
    """Calculate overall run expenses using current pricing metrics per million tokens."""
    pricing = PRICING_PER_1M.get(model, {"input": 0.0, "output": 0.0})
    input_tokens = estimate_tokens(input_text)
    output_tokens = estimate_tokens(output_text)

    # Calculate input and output costs separately and combine them
    cost = ((input_tokens / 1_000_000) * pricing["input"]) + \
           ((output_tokens / 1_000_000) * pricing["output"])

    return input_tokens, output_tokens, round(cost, 8)

print("Token estimation test:", estimate_tokens("Hello students"))

Token estimation test: 4


### **7. Exponential Backoff and Retry Helper**
To build a highly resilient pipeline, we implement `run_with_retry`. This function wraps any call and automatically retries upon failure using **Exponential Backoff** with a random jitter (delay increases exponentially: 1s, 2s, 4s, etc.), preventing API rate-limiting stampedes.

In [ ]:
def run_with_retry(fn, max_retries: int = 3, base_delay: float = 1.0):
    """Execute a function with exponential backoff and jitter to mitigate transient faults."""
    attempt = 0
    last_error = None

    while attempt < max_retries:
        try:
            attempt += 1
            result = fn()
            return result, attempt
        except Exception as e:
            last_error = str(e)

            if attempt >= max_retries:
                break

            # Exponential delay formula: base * (2^(attempt-1)) + random jitter
            wait_time = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.3)
            print(f"Attempt {attempt} failed: {e}")
            print(f"Retrying in {wait_time:.2f} seconds...")
            time.sleep(wait_time)

    raise RuntimeError(f"Failed after {max_retries} attempts. Last error: {last_error}")

print("Retry helper created.")

Retry helper created.


### **8. Implement the OpenAI Provider Class**
This class encapsulates interaction with OpenAI:
- `stream_text`: Uses the SDK to request a streamed chat completion and dynamically prints text tokens to the stdout as they arrive.
- `generate`: Wraps the streaming logic inside our retry helper, gathers execution telemetry, and packs everything neatly into a `GatewayResult` object.

In [ ]:
class OpenAIProvider:
    """Provider class handling execution and streaming from the OpenAI client."""
    def __init__(self, api_key: str):
        if not api_key:
            raise ValueError("OpenAI API key is missing.")
        # Initialize the underlying OpenAI client using standard client initialization
        self.client = OpenAI(api_key=api_key)

    def stream_text(self, prompt: str, model: str, temperature: float = 0.2) -> str:
        """Helper method to handle streaming API chunks directly to standard output."""
        collected_text = []

        # Call the chat completions endpoint with stream enabled
        stream = self.client.responses.create(
            model=model,
            input=prompt,
            temperature=temperature,
            stream=True
        )

        print("Streaming response:\n")
        for event in stream:
            event_type = getattr(event, "type", "")

            # Capture chunk updates emitted during stream progression
            if event_type == "response.output_text.delta":
                delta = getattr(event, "delta", "")
                if delta:
                    print(delta, end="", flush=True)
                    collected_text.append(delta)

        print("\n\nStreaming completed.")
        return "".join(collected_text)

    def generate(self, request: GatewayRequest, model: str, provider_requested: str) -> GatewayResult:
        """Wraps the streamed execution with our backoff retry and cost metrics logging."""
        def _call():
            return self.stream_text(
                prompt=request.prompt,
                model=model,
                temperature=request.temperature
            )

        # Attempt to run with retry policy
        output_text, attempts = run_with_retry(_call, max_retries=request.max_retries)
        input_tokens, output_tokens, cost = estimate_cost(model, request.prompt, output_text)

        return GatewayResult(
            provider_requested=provider_requested,
            provider_used="openai",
            model=model,
            text=output_text,
            estimated_input_tokens=input_tokens,
            estimated_output_tokens=output_tokens,
            estimated_cost_usd=cost,
            attempts=attempts,
            success=True
        )

### **9. Construct the Multi-Provider Gateway**
The `MultiProviderGateway` orchestrates routing. If a task's preferred provider is not available or supported locally (such as Anthropic or local Ollama), it safely logs a fallback notice and redirects the request to the functional OpenAI provider, ensuring high availability.

In [ ]:
class MultiProviderGateway:
    """Gateway class coordinating request routing and execution fallbacks."""
    def __init__(self, openai_api_key: str):
        # Instantiate our primary OpenAI executor
        self.openai_provider = OpenAIProvider(openai_api_key)
        self.anthropic_provider = AnthropicProvider(anthropic_api_key)
        self.google_provider =
        self.ollama_provider =

    def route(self, task_type: str) -> Dict[str, str]:
        """Look up routing instructions within the static ROUTING_TABLE dictionary."""
        if task_type not in ROUTING_TABLE:
            raise ValueError(f"Unsupported task_type: {task_type}")
        return ROUTING_TABLE[task_type]

    def run(self, request: GatewayRequest) -> GatewayResult:
        """Main entry point: routes request, triggers fallback if needed, and executes."""
        route_info = self.route(request.task_type)
        preferred_provider = route_info["preferred_provider"]
        model = route_info["model"]

        # Since only OpenAI is active in this environment, apply fallback redirection logic
        if preferred_provider != "openai":
            print(f"Preferred provider '{preferred_provider}' is not available in this lab.")
            print("Falling back to OpenAI.\n")

        return self.openai_provider.generate(
            request=request,
            model=model,
            provider_requested=preferred_provider
        )

# Instantiate the main gateway object using our stored environment variables
gateway = MultiProviderGateway(openai_api_key=os.environ["OPENAI_API_KEY"])
print("Gateway initialized successfully.")

Gateway initialized successfully.


### **10. Execute Test Case: Chat Request**
We initialize our gateway and trigger our first test request of task type `'chat'`. This demonstrates streaming response generation from the OpenAI backend.

In [ ]:
# Define a basic chat query using standard configuration settings
request_1 = GatewayRequest(
    prompt="Explain in 5 short points why an LLM gateway is useful in production systems.",
    task_type="chat",
    stream=True,
    max_retries=3,
    temperature=0.2
)

# Run the request via our multi-provider gateway instance
result_1 = gateway.run(request_1)

Streaming response:

1. **Scalability**: LLM gateways can efficiently manage multiple requests, allowing systems to scale seamlessly as user demand increases.

2. **Load Balancing**: They distribute incoming traffic across multiple LLM instances, optimizing resource utilization and ensuring consistent performance.

3. **Security**: Gateways can implement authentication and authorization, protecting sensitive data and ensuring only authorized users access the LLM.

4. **Monitoring and Analytics**: They provide insights into usage patterns and performance metrics, enabling better decision-making and system improvements.

5. **Version Control**: LLM gateways facilitate easy updates and version management, allowing for smooth transitions between different model versions without disrupting service.

Streaming completed.


### **11. Display Execution Metadata**
We inspect the output object (`result_1`) to verify that the gateway logged metadata correctly, including attempts, generated text, token estimations, and calculated costs.

In [ ]:
# Print out the collected performance telemetry for analysis
print("\n--- Final Result ---")
print("Provider Requested:", result_1.provider_requested)
print("Provider Used:", result_1.provider_used)
print("Model:", result_1.model)
print("Attempts:", result_1.attempts)
print("Estimated Input Tokens:", result_1.estimated_input_tokens)
print("Estimated Output Tokens:", result_1.estimated_output_tokens)
print("Estimated Cost (USD):", result_1.estimated_cost_usd)
print("\nFinal Text:\n")
print(result_1.text)


--- Final Result ---
Provider Requested: openai
Provider Used: openai
Model: gpt-4o-mini
Attempts: 1
Estimated Input Tokens: 20
Estimated Output Tokens: 196
Estimated Cost (USD): 0.0001206

Final Text:

1. **Scalability**: LLM gateways can efficiently manage multiple requests, allowing systems to scale seamlessly as user demand increases.

2. **Load Balancing**: They distribute incoming traffic across multiple LLM instances, optimizing resource utilization and ensuring consistent performance.

3. **Security**: Gateways can implement authentication and authorization, protecting sensitive data and ensuring only authorized users access the LLM.

4. **Monitoring and Analytics**: They provide insights into usage patterns and performance metrics, enabling better decision-making and system improvements.

5. **Version Control**: LLM gateways facilitate easy updates and version management, allowing for smooth transitions between different model versions without disrupting service.


### **12. Execute Test Case: Coding Request**
We send a second request configured for a `'coding'` task. The routing table maps this to `'openai'` and `'gpt-4o-mini'`, delivering streaming code instructions directly to the notebook.

In [ ]:
# Define a technical coding prompt targeted for specific model processing
request_2 = GatewayRequest(
    prompt="Write a Python function to check whether a number is even or odd and explain it in 3 short points.",
    task_type="coding",
    stream=True,
    max_retries=3,
    temperature=0.1
)

# Run coding task routing directly through the gateway
result_2 = gateway.run(request_2)

Streaming response:

Here's a simple Python function to check whether a number is even or odd:

```python
def is_even_or_odd(number):
    if number % 2 == 0:
        return "Even"
    else:
        return "Odd"
```

### Explanation:
1. **Modulo Operator**: The function uses the modulo operator `%` to determine the remainder when the number is divided by 2. If the remainder is 0, the number is even.
  
2. **Conditional Statement**: An `if` statement checks the result of the modulo operation. If it equals 0, it returns "Even"; otherwise, it returns "Odd".

3. **Return Value**: The function returns a string indicating whether the input number is "Even" or "Odd", making it easy to understand the result.

Streaming completed.


### **13. Execute Test Case: Analysis (Fallback Scenario)**
This test requests an `'analysis'` task. The routing table lists `'anthropic'` as the preferred provider. Since it is unavailable, the gateway falls back automatically to OpenAI, showcasing the routing system's reliability.

In [ ]:
# Trigger a request mapped to Anthropic under the hood to evaluate routing fallback rules
request_3 = GatewayRequest(
    prompt="Give a short analysis of why exponential backoff is useful for API retry logic.",
    task_type="analysis",
    stream=True,
    max_retries=3,
    temperature=0.2
)

# The system flags Anthropic's absence and reroutes to the active OpenAI provider
result_3 = gateway.run(request_3)

Preferred provider 'anthropic' is not available in this lab.
Falling back to OpenAI.

Streaming response:

Exponential backoff is a strategy used in API retry logic that involves progressively increasing the wait time between successive retries after a failure. Here are some key reasons why it is useful:

1. **Avoids Overloading the Server**: By increasing the wait time between retries, exponential backoff helps prevent overwhelming the server with repeated requests, which can exacerbate the problem and lead to further failures.

2. **Network Congestion Management**: In scenarios where network issues are causing failures, exponential backoff allows the system to reduce the frequency of requests, giving time for the network to stabilize.

3. **Improved Success Rates**: By spacing out retries, the likelihood of a successful request increases, especially if the failure is due to temporary issues like rate limiting or server overload.

4. **Fairness**: This approach ensures that all client

### **14. Execute Test Case: Private Local (Fallback Scenario)**
This test requests a `'private_local'` task. The routing table lists `'ollama'` as preferred. As Ollama is not installed locally, the gateway gracefully falls back to OpenAI once again.

In [ ]:
# Trigger a task designated for local on-device execution (ollama preferred provider)
request_4 = GatewayRequest(
    prompt="Summarize the benefits of local model inference in 4 short points.",
    task_type="private_local",
    stream=True,
    max_retries=3,
    temperature=0.2
)

# Verify fallback to central cloud gateway API
result_4 = gateway.run(request_4)

Preferred provider 'ollama' is not available in this lab.
Falling back to OpenAI.

Streaming response:

1. **Reduced Latency**: Local model inference minimizes response time by processing data on-device, eliminating delays from network communication.

2. **Enhanced Privacy**: Sensitive data remains on the device, reducing the risk of exposure and ensuring user privacy.

3. **Offline Functionality**: Local inference allows applications to function without internet access, providing continuous usability in various environments.

4. **Lower Bandwidth Usage**: By processing data locally, it decreases the need for data transmission, saving bandwidth and reducing costs.

Streaming completed.


### **15. Aggregate and Review Gateway Run Telemetry**
We assemble all previous execution results to compare the requested provider versus the actual provider used, highlighting how the gateway managed fallback transitions seamlessly.

In [ ]:
# Aggregate all execution result instances to compare planned vs fallback routes
results = [result_1, result_2, result_3, result_4]

for i, res in enumerate(results, start=1):
    print(f"\n--- Request {i} Summary ---")
    print("Provider Requested:", res.provider_requested)
    print("Provider Used:", res.provider_used)
    print("Model:", res.model)
    print("Attempts:", res.attempts)
    print("Estimated Cost (USD):", res.estimated_cost_usd)


--- Request 1 Summary ---
Provider Requested: openai
Provider Used: openai
Model: gpt-4o-mini
Attempts: 1
Estimated Cost (USD): 0.0001206

--- Request 2 Summary ---
Provider Requested: openai
Provider Used: openai
Model: gpt-4o-mini
Attempts: 1
Estimated Cost (USD): 0.00010695

--- Request 3 Summary ---
Provider Requested: anthropic
Provider Used: openai
Model: gpt-4o-mini
Attempts: 1
Estimated Cost (USD): 0.000207

--- Request 4 Summary ---
Provider Requested: ollama
Provider Used: openai
Model: gpt-4o-mini
Attempts: 1
Estimated Cost (USD): 8.775e-05


**Lab 2: Create a structured data extraction pipeline using Instructor + Pydantic**

### **16. Install Structured Extraction Libraries (Lab 2)**
We install the `instructor` library alongside `openai` and `pydantic`. Instructor leverages Pydantic schemas to enforce strict, structured JSON outputs from LLMs.

In [ ]:
# Upgrade/install pip modules to enable robust structured extraction in Python
!pip -q install --upgrade instructor openai pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 19.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.


### **17. Load API Credentials securely**
We ensure that the required credentials are correctly loaded from the Secrets manager to authorize Instructor API requests.

In [ ]:
import os
from google.colab import userdata

# Read credentials securely to authenticate schema extraction clients
OPENAI_API_KEY = userdata.get("APIKEY")

if not OPENAI_API_KEY:
    raise ValueError("API key not found. Add a secret named 'APIKEY' and enable access.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("API key loaded successfully.")

API key loaded successfully.


### **18. Import Structured Extraction Modules**
We import the schema validation utilities, including `instructor` and Pydantic's foundational schema constructs (`BaseModel`, `Field`, `ValidationError`).

In [ ]:
import os
import json
import time
import random
from typing import Optional, List, Literal

# Import Instructor and Pydantic validation tools
import instructor
from pydantic import BaseModel, Field, ValidationError, ConfigDict

### **19. Initialize the Instructor Client**
We wrap an OpenAI model configuration inside the Instructor client using `instructor.from_provider()`. This configuration applies OpenAI function calling tool mechanics (`RESPONSES_TOOLS`) behind the scenes to enforce structural schema boundaries.

In [ ]:
# Create an extended client using Instructor wrapper with openai backing
# RESPONSES_TOOLS leverages the provider's native JSON schema tool calling feature
client = instructor.from_provider(
    "openai/gpt-4.1-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    mode=instructor.Mode.RESPONSES_TOOLS
)

print("Instructor client created successfully.")

Instructor client created successfully.


### **20. Model the Target Product Schema**
We define a Pydantic `ProductSpec` model subclassing `BaseModel`. Each attribute includes a `Field` description detailing what data should be extracted. We enforce strict literal boundaries on categories, positive numeric limits on product attributes, and prohibit extra fields (`extra='forbid'`) to keep the JSON clean.

In [ ]:
class ProductSpec(BaseModel):
    """Schema definitions specifying precise structural bounds for extraction."""
    # Restrict response output strictly to fields specified in class attributes
    model_config = ConfigDict(extra="forbid")

    product_name: str = Field(description="Full product name as mentioned in the description")
    brand: str = Field(description="Brand name of the product")
    # Enforce strict categorization values using Literal typing constraint
    category: Literal["smartphone", "laptop", "headphones", "tablet", "smartwatch", "camera", "other"] = Field(
        description="Best matching product category"
    )
    color: Optional[str] = Field(default=None, description="Product color if present")
    storage_gb: Optional[int] = Field(default=None, ge=1, description="Storage capacity in GB if available")
    ram_gb: Optional[int] = Field(default=None, ge=1, description="RAM size in GB if available")
    battery_mah: Optional[int] = Field(default=None, ge=100, description="Battery size in mAh if available")
    screen_size_inches: Optional[float] = Field(default=None, gt=0, description="Screen size in inches if available")
    price: Optional[float] = Field(default=None, ge=0, description="Numeric price only, without currency symbols")
    currency: Optional[str] = Field(default=None, description="Currency code or symbol if available")
    in_stock: Optional[bool] = Field(default=None, description="Whether the item is available or in stock")
    key_features: List[str] = Field(default_factory=list, description="Short list of important product features")

### **21. Setup Sample Unstructured Dataset**
We build an array of multi-line strings mimicking unstructured product listings of varying structures, containing different brands, currencies, formats, and levels of detail.

In [ ]:
# Prepare raw, mock product logs to evaluate parsing accuracy
product_descriptions = [
    """
    Apple iPhone 15, Blue, 128GB storage. Super Retina XDR display, advanced dual-camera system,
    all-day battery life, USB-C charging. Price: $799. Currently in stock.
    """,

    """
    Samsung Galaxy S24 Ultra smartphone in Titanium Black with 256GB internal storage and 12GB RAM.
    Features a 6.8-inch display, 5000mAh battery, AI camera enhancements, and fast charging.
    Listed at 1299 USD. Available now.
    """,

    """
    Lenovo IdeaPad Slim 5 laptop, 16GB RAM, 512GB SSD, 15.6 inch display, Cloud Grey finish.
    Built for productivity and multitasking. Starting price INR 64999. Limited stock.
    """
]

print("Loaded", len(product_descriptions), "sample product descriptions.")

Loaded 3 sample product descriptions.


### **22. Construct the Extraction Engine Function**
We write `extract_product_spec` to handle single-item extraction. By passing the Pydantic class `ProductSpec` to `response_model`, Instructor validates that the returned JSON matches our schema definitions precisely before presenting the final object.

In [ ]:
def extract_product_spec(description: str) -> ProductSpec:
    """Extract unstructured description into validation-verified ProductSpec schema object."""
    result = client.responses.create(
        input=[
            {
                "role": "system",
                "content": (
                    "You extract product specifications from e-commerce text. "
                    "Return only values supported by the text. "
                    "Do not guess missing values. "
                    "Use null for missing optional fields."
                ),
            },
            {
                "role": "user",
                "content": f"Extract the product specifications from this description:\n\n{description}",
            },
        ],
        # The response_model config forces structural parsing checks against our Pydantic class
        response_model=ProductSpec,
    )
    return result

### **23. Run Extraction Smoke Test**
We run a baseline validation check by extracting details from the first product description. We verify both the structured model response and its equivalent dictionary serialization.

In [ ]:
# Run a basic parsing validation smoke test over our first array listing
sample_result = extract_product_spec(product_descriptions[0])

print(sample_result)
print("\nAs dictionary:")
# Convert Pydantic object to a standard Python dictionary
print(sample_result.model_dump())

product_name='Apple iPhone 15' brand='Apple' category='smartphone' color='Blue' storage_gb=128 ram_gb=None battery_mah=None screen_size_inches=None price=799.0 currency='$' in_stock=True key_features=['Super Retina XDR display', 'advanced dual-camera system', 'all-day battery life', 'USB-C charging']

As dictionary:
{'product_name': 'Apple iPhone 15', 'brand': 'Apple', 'category': 'smartphone', 'color': 'Blue', 'storage_gb': 128, 'ram_gb': None, 'battery_mah': None, 'screen_size_inches': None, 'price': 799.0, 'currency': '$', 'in_stock': True, 'key_features': ['Super Retina XDR display', 'advanced dual-camera system', 'all-day battery life', 'USB-C charging']}


### **24. Implement Robust Validation Retries**
Because structured extraction can fail due to schema mismatches (`ValidationError`) or transient connection glitches, we implement a specialized retry loop that applies exponential backoff, maximizing extraction resilience.

In [ ]:
def extract_with_retry(description: str, max_retries: int = 3, base_delay: float = 1.0) -> ProductSpec:
    """Run structural extraction with localized error catching and retry policies."""
    attempt = 0
    last_error = None

    while attempt < max_retries:
        try:
            attempt += 1
            return extract_product_spec(description)
        except ValidationError as ve:
            # Handle Pydantic type validation mismatches cleanly
            last_error = f"ValidationError: {ve}"
            print(f"Attempt {attempt} failed due to schema validation.")
        except Exception as e:
            # Handle API network connectivity errors gracefully
            last_error = f"Exception: {e}"
            print(f"Attempt {attempt} failed due to request or parsing error: {e}")

        if attempt < max_retries:
            # Add randomized Jitter delay calculation
            delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.3)
            print(f"Retrying in {delay:.2f} seconds...\n")
            time.sleep(delay)

    raise RuntimeError(f"Extraction failed after {max_retries} attempts. Last error: {last_error}")

### **25. Process the Entire Pipeline**
We run our robust extraction pipeline over all unstructured input descriptions, extracting clean, structured datasets and catching schemas seamlessly.

In [ ]:
# Iterate over every unstructured description and generate validated objects
structured_products = []

for i, description in enumerate(product_descriptions, start=1):
    print(f"\n--- Extracting Product {i} ---")
    product = extract_with_retry(description, max_retries=3)
    structured_products.append(product)
    print("Extraction successful.")
    print(product.model_dump())


--- Extracting Product 1 ---
Extraction successful.
{'product_name': 'Apple iPhone 15', 'brand': 'Apple', 'category': 'smartphone', 'color': 'Blue', 'storage_gb': 128, 'ram_gb': None, 'battery_mah': None, 'screen_size_inches': None, 'price': 799.0, 'currency': '$', 'in_stock': True, 'key_features': ['Super Retina XDR display', 'advanced dual-camera system', 'all-day battery life', 'USB-C charging']}

--- Extracting Product 2 ---
Extraction successful.
{'product_name': 'Samsung Galaxy S24 Ultra', 'brand': 'Samsung', 'category': 'smartphone', 'color': 'Titanium Black', 'storage_gb': 256, 'ram_gb': 12, 'battery_mah': 5000, 'screen_size_inches': 6.8, 'price': 1299.0, 'currency': 'USD', 'in_stock': True, 'key_features': ['AI camera enhancements', 'fast charging']}

--- Extracting Product 3 ---
Extraction successful.
{'product_name': 'Lenovo IdeaPad Slim 5 laptop', 'brand': 'Lenovo', 'category': 'laptop', 'color': 'Cloud Grey', 'storage_gb': 512, 'ram_gb': 16, 'battery_mah': None, 'screen_s

### **26. Serialize Output to JSON Format**
Using Pydantic's `.model_dump()` method, we serialize our Python objects into Python dictionaries and convert the final dataset into a clean, human-readable JSON block.

In [ ]:
# Serialize our list of validated instances into a list of pure dict objects
json_ready_output = [product.model_dump() for product in structured_products]

# Convert to pretty-printed JSON string formatting
print(json.dumps(json_ready_output, indent=2))

[
  {
    "product_name": "Apple iPhone 15",
    "brand": "Apple",
    "category": "smartphone",
    "color": "Blue",
    "storage_gb": 128,
    "ram_gb": null,
    "battery_mah": null,
    "screen_size_inches": null,
    "price": 799.0,
    "currency": "$",
    "in_stock": true,
    "key_features": [
      "Super Retina XDR display",
      "advanced dual-camera system",
      "all-day battery life",
      "USB-C charging"
    ]
  },
  {
    "product_name": "Samsung Galaxy S24 Ultra",
    "brand": "Samsung",
    "category": "smartphone",
    "color": "Titanium Black",
    "storage_gb": 256,
    "ram_gb": 12,
    "battery_mah": 5000,
    "screen_size_inches": 6.8,
    "price": 1299.0,
    "currency": "USD",
    "in_stock": true,
    "key_features": [
      "AI camera enhancements",
      "fast charging"
    ]
  },
  {
    "product_name": "Lenovo IdeaPad Slim 5 laptop",
    "brand": "Lenovo",
    "category": "laptop",
    "color": "Cloud Grey",
    "storage_gb": 512,
    "ram_gb": 16,
  

### **27. Write Dataset to Disk**
We save the extracted and serialized JSON array to a local file named `structured_product_specs.json` on the disk.

In [ ]:
# Save formatted JSON structure to local disk storage
with open("structured_product_specs.json", "w", encoding="utf-8") as f:
    json.dump(json_ready_output, f, indent=2, ensure_ascii=False)

print("File saved as structured_product_specs.json")

File saved as structured_product_specs.json


### **28. Test Validation Error Resiliency**
We feed a non-compliant, highly abstract text into the pipeline to check how the parser reacts. When fields cannot be extracted, the model intelligently defaults optional parameters to `None` rather than raising a parsing exception, proving its structural stability.

In [ ]:
# Prepare an abstract input text designed to challenge extraction accuracy
bad_description = """
Amazing premium device with next-generation technology, stylish body, excellent experience,
great performance, and stunning design. Best choice for modern users.
"""

try:
    # Process non-compliant text and see how well Pydantic handles defaults
    bad_result = extract_with_retry(bad_description, max_retries=2)
    print(bad_result.model_dump())
except Exception as e:
    print("Handled pipeline error:")
    print(e)

{'product_name': 'Amazing premium device', 'brand': '', 'category': 'other', 'color': None, 'storage_gb': None, 'ram_gb': None, 'battery_mah': None, 'screen_size_inches': None, 'price': None, 'currency': None, 'in_stock': None, 'key_features': ['next-generation technology', 'stylish body', 'excellent experience', 'great performance', 'stunning design', 'best choice for modern users']}


### **29. Encapsulate Pipeline as a Reusable Function**
We group the extraction, retry, and dictionary dumping steps into `run_extraction_pipeline` so that the extraction logic can easily be executed on new, larger lists of raw product descriptions.

In [ ]:
def run_extraction_pipeline(descriptions: List[str]) -> List[dict]:
    """Helper to batch process unstructured product lists cleanly."""
    outputs = []

    for idx, description in enumerate(descriptions, start=1):
        print(f"\nProcessing item {idx}...")
        # Execute standard robust extraction for each item
        product = extract_with_retry(description, max_retries=3)
        outputs.append(product.model_dump())

    return outputs

### **30. Execute the Reusable Pipeline**
We run a final test of the encapsulated pipeline on our list of descriptions, ensuring it executes efficiently and yields the expected structured list output.

In [ ]:
# Verify execution of the newly encapsulated function on our sample dataset
final_output = run_extraction_pipeline(product_descriptions)

print("\nFinal structured output:")
print(json.dumps(final_output, indent=2))


Processing item 1...

Processing item 2...

Processing item 3...

Final structured output:
[
  {
    "product_name": "Apple iPhone 15",
    "brand": "Apple",
    "category": "smartphone",
    "color": "Blue",
    "storage_gb": 128,
    "ram_gb": null,
    "battery_mah": null,
    "screen_size_inches": null,
    "price": 799.0,
    "currency": "$",
    "in_stock": true,
    "key_features": [
      "Super Retina XDR display",
      "advanced dual-camera system",
      "all-day battery life",
      "USB-C charging"
    ]
  },
  {
    "product_name": "Samsung Galaxy S24 Ultra",
    "brand": "Samsung",
    "category": "smartphone",
    "color": "Titanium Black",
    "storage_gb": 256,
    "ram_gb": 12,
    "battery_mah": 5000,
    "screen_size_inches": 6.8,
    "price": 1299.0,
    "currency": "USD",
    "in_stock": true,
    "key_features": [
      "6.8-inch display",
      "5000mAh battery",
      "AI camera enhancements",
      "fast charging"
    ]
  },
  {
    "product_name": "Lenovo 